In [1]:
import torch
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score
from core import *
from utils import *
from lark import Tree, Token
from pm4py import save_vis_petri_net
import random
import pandas as pd
import sys
#print(f"Versione Python: {sys.version}")

#print(torch.cuda.is_available())
#print(torch.__version__)

# SETTINGS
NARY = 1
PROBABILITIES = 0.2,0.2,0.2,0.4
FILE_PATH_PNG = "petri_net_output.png"
TRACE_ENC_REG = "data/regions_full.csv"
TRACE_ENC_TAS = "data/tasks_full.csv"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
learning_rate = 3e-4

cuda


In [2]:
def get_batch_model(split, train_data, val_data, batch_size, block_size, device):
    data = train_data if split == 'train' else val_data

    ix = torch.randint(len(data) - block_size, (batch_size,)) # Prendo batch_size indici casuali

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    x,y = x.to(device), y.to(device)
    return x, y

# Get batch per il transformer del time a cui diamo lo 0 al posto di fare lo sliding (al momento non utilizzato)
'''def get_batch_model_time(split, train_data, train_times, val_data, val_times, batch_size, block_size, device):
    data = train_data if split == 'train' else val_data
    time = train_times if split == 'train' else val_times

    ix = torch.randint(len(data) - block_size, (batch_size,))

    x_data = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([time[i:i+block_size] for i in ix])

    x_times = y.clone()
    x_times[:, -1] = 0.0

    x_data, x_times, y = x_data.to(device), x_times.to(device), y.to(device)
    return x_data, x_times, y'''

# VERSIONE 2 - IPOTIZZO SIA MEGLIO EH
def get_batch_model_time(split, train_data, train_times, val_data, val_times, batch_size, block_size, device):
    data = train_data if split == 'train' else val_data
    time = train_times if split == 'train' else val_times

    ix = torch.randint(1, len(data) - block_size, (batch_size,))

    x_data = torch.stack([data[i:i+block_size] for i in ix])
    x_times = torch.stack([time[i-1:i+block_size-1] for i in ix])
    y = torch.stack([time[i:i+block_size] for i in ix])

    x_data, x_times, y = x_data.to(device), x_times.to(device), y.to(device)
    return x_data, x_times, y

def get_batch_model_time_v2(split, train_task, train_region, train_times, val_task, val_region, val_times, batch_size, block_size, device):
    task = train_task if split == 'train' else val_task
    region = train_region if split == 'train' else val_region
    time = train_times if split == 'train' else val_times

    ix = torch.randint(1, len(task) - block_size, (batch_size,))

    x_task = torch.stack([task[i:i+block_size] for i in ix])
    x_region = torch.stack([region[i:i+block_size] for i in ix])
    x_times = torch.stack([time[i-1:i+block_size-1] for i in ix])
    y = torch.stack([time[i:i+block_size] for i in ix])

    x_task, x_region, x_times, y = x_task.to(device), x_region.to(device), x_times.to(device), y.to(device)
    return x_task, x_region, x_times, y

def get_batch_model_region(split, train_task, val_task, train_region, val_region, batch_size, block_size, device):
    task = train_task if split == 'train' else val_task
    region = train_region if split == 'train' else val_region

    ix = torch.randint(1, len(task) - block_size, (batch_size,))

    x_task = torch.stack([task[i:i+block_size] for i in ix])
    x_region = torch.stack([region[i-1:i+block_size-1] for i in ix])
    y = torch.stack([region[i:i+block_size] for i in ix])

    x_task, x_region, y = x_task.to(device), x_region.to(device), y.to(device)
    return x_task, x_region, y

@torch.no_grad()
def estimate_loss(model, eval_iters, train_data, val_data, batch_size, block_size, device):
    out = {}
    model.eval() # Metto il modello in modalità eval (non train)

    # Calcolo la loss su eval_iters batch
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch_model(split, train_data, val_data, batch_size, block_size, device)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()

    model.train()
    return out

@torch.no_grad()
def estimate_loss_region(model, eval_iters, train_task, train_region, val_task, val_region, batch_size, block_size, device):
    out = {}
    model.eval() # Metto il modello in modalità eval (non train)

    # Calcolo la loss su eval_iters batch
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X_task, X_region, Y = get_batch_model_region(split, train_task, val_task, train_region, val_region, batch_size, block_size, device)
            logits, loss = model(X_region, X_task, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()

    model.train()
    return out

@torch.no_grad()
def estimate_loss_times(model, eval_iters, train_data, train_times, val_data, val_times, batch_size, block_size, device):
    out = {}
    model.eval() # Metto il modello in modalità eval (non train)

    # Calcolo la loss su eval_iters batch
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X_data, X_times, Y = get_batch_model_time(split, train_data, train_times, val_data, val_times, batch_size, block_size, device)
            logits, loss = model(X_data, X_times, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()

    model.train()
    return out

@torch.no_grad()
def estimate_loss_times_v2(model, eval_iters, train_task, train_region, train_times, val_task, val_region, val_times, batch_size, block_size, device):
    out = {}
    model.eval() # Metto il modello in modalità eval (non train)

    # Calcolo la loss su eval_iters batch
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X_task, X_region, X_times, Y = get_batch_model_time_v2(split, train_task, train_region, train_times, val_task, val_region, val_times, batch_size, block_size, device)
            logits, loss = model(X_task, X_region, X_times, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()

    model.train()
    return out

In [3]:
iterations = 15  # Quante iterazioni diverse (numero regioni prima di minimizzare in teoria)
current_string = SEED_STRING
for _ in range(iterations):
    current_string = replace_random_underscore(current_string, PROBABILITIES)

process = replace_underscores(current_string)
tree = PARSER.parse(process)

# ALBERI GIOCATTOLO

tree = Tree('xor', [Tree('task', [Token('NAME', 'T1')]),Tree('sequential', [Tree('parallel', [Tree('xor', [Tree('task', [Token('NAME', 'T2')]),Tree('xor', [Tree('task', [Token('NAME', 'T3')]),Tree('task', [Token('NAME', 'T4')])])]),Tree('parallel', [Tree('task', [Token('NAME', 'T5')]),Tree('task', [Token('NAME', 'T6')])])]),Tree('sequential', [Tree('task', [Token('NAME', 'T7')]),Tree('task', [Token('NAME', 'T8')])])])])

#tree = Tree('xor', [Tree('loop', [Tree('sequential', [Tree('loop', [Tree('task', [Token('NAME', 'T1')])]), Tree('xor', [Tree('task', [Token('NAME', 'T2')]), Tree('task', [Token('NAME', 'T3')])])])]), Tree('parallel', [Tree('task', [Token('NAME', 'T4')]), Tree('loop', [Tree('task', [Token('NAME', 'T5')])])])])

if NARY: # Se true allora faccio l'albero ennario (semplifico)
    tree = createNAryTree(tree)

#tree = Tree('loop', [Tree('parallel', [Tree('sequential', [Tree('loop', [Tree('parallel', [Tree('task', [Token('NAME', 'T1')]), Tree('task', [Token('NAME', 'T2')])])]), Tree('loop', [Tree('parallel', [Tree('loop', [Tree('task', [Token('NAME', 'T3')])]), Tree('task', [Token('NAME', 'T4')])])])]), Tree('loop', [Tree('sequential', [Tree('task', [Token('NAME', 'T5')]), Tree('loop', [Tree('task', [Token('NAME', 'T6')])])])])])])

print(tree)

Tree('xor', [Tree('task', [Token('NAME', 'T1')]), Tree('sequential', [Tree('parallel', [Tree('xor', [Tree('task', [Token('NAME', 'T2')]), Tree('task', [Token('NAME', 'T3')]), Tree('task', [Token('NAME', 'T4')])]), Tree('task', [Token('NAME', 'T5')]), Tree('task', [Token('NAME', 'T6')])]), Tree('task', [Token('NAME', 'T7')]), Tree('task', [Token('NAME', 'T8')])])])


In [4]:
net = PetriNetP(tree)

save_vis_petri_net(
    net.net,
    net.initial_marking,
    net.final_marking,
    FILE_PATH_PNG,
    format="png"
)

# Oggetto Generator
generator = Generator(5000, net)

In [5]:
# Creazione matrice identità delle regioni
df_region_identity = pd.DataFrame.from_dict(net.node_identity, orient='index').sort_index()
df_region_identity.columns = ['X', '+', '->', '<>']
print(df_region_identity)

# Creazione matrice regioni-figli per le regioni
df_region_children = pd.Series(net.node_children).explode()
df_region_children = pd.crosstab(df_region_children.index, df_region_children)
df_region_children = df_region_children.reindex(index=net.regions, columns=net.regions + net.tasks, fill_value=0)
df_region_children = df_region_children.astype(int)
df_region_children.index.name = None
df_region_children.columns.name = None
print(df_region_children)

# Codifica delle tracce generate
traceEncoded_regions, traceEncoded_tasks = getEncoding(generator.generatedTraces, net.regions, net.tasks, net.open_clauses, net.end_clauses)

# Trovo numero regioni e numero task effettivo (torneranno utili in futuro)
num_regions = len([i for i in traceEncoded_regions.index if str(i).startswith('R')])
num_tasks = len([i for i in traceEncoded_tasks.index if str(i).startswith('T')])

# Penso evitabili d'ora in poi
#traceEncoded_regions.to_csv(TRACE_ENC_REG, index=True)
#traceEncoded_tasks.to_csv(TRACE_ENC_TAS, index=True)

# Creo il dataframe unico (regioni + tasks)
df_traces_complete = pd.concat([traceEncoded_regions, traceEncoded_tasks], axis=0)
print(df_traces_complete)

# Operazioni per creazione funzione di codifica e decodifica da colonna ad id e viceversa (per regione + task)
df_traces_complete = df_traces_complete.T
df_tracescopy = df_traces_complete.copy()

unique_columns_complete = df_traces_complete.drop_duplicates()
unique_tuple_complete = [tuple(x) for x in unique_columns_complete.values]

bit_to_id_traces_complete = {v: i for i, v in enumerate(unique_tuple_complete)}
id_to_bit_traces_complete = {i: v for i, v in enumerate(unique_tuple_complete)}

vocab_size_complete = len(unique_columns_complete)

encode_complete = lambda a: [bit_to_id_traces_complete[tuple(x)] for x in a]
decode_complete = lambda b: [id_to_bit_traces_complete[x] for x in b]

# Operazioni per creazione funzione di codifica e decodifica da colonna ad id e viceversa (solo per REGIONE)
df_regions = traceEncoded_regions.copy()
df_regions = df_regions.T

unique_columns_regions = df_regions.drop_duplicates()
unique_tuple_regions = [tuple(x) for x in unique_columns_regions.values]

bit_to_id_traces_regions = {v: i for i, v in enumerate(unique_tuple_regions)}
id_to_bit_traces_regions = {i: v for i, v in enumerate(unique_tuple_regions)}

vocab_size_regions = len(unique_columns_regions)

encode_regions = lambda a: [bit_to_id_traces_regions[tuple(x)] for x in a]
decode_regions = lambda b: [id_to_bit_traces_regions[x] for x in b]

# Operazioni per creazione funzione di codifica e decodifica da colonna ad id e viceversa (solo per TASK)
df_tasks = traceEncoded_tasks.copy()
df_tasks = df_tasks.T

unique_columns_tasks = df_tasks.drop_duplicates()
unique_tuple_tasks = [tuple(x) for x in unique_columns_tasks.values]

bit_to_id_traces_tasks = {v: i for i, v in enumerate(unique_tuple_tasks)}
id_to_bit_traces_tasks = {i: v for i, v in enumerate(unique_tuple_tasks)}

vocab_size_tasks = len(unique_columns_tasks)

encode_tasks = lambda a: [bit_to_id_traces_tasks[tuple(x)] for x in a]
decode_tasks = lambda b: [id_to_bit_traces_tasks[x] for x in b]

    X  +  ->  <>
R0  1  0   0   0
R1  0  0   1   0
R2  0  1   0   0
R3  1  0   0   0
    R0  R1  R2  R3  T1  T2  T3  T4  T5  T6  T7  T8
R0   0   1   0   0   1   0   0   0   0   0   0   0
R1   0   0   1   0   0   0   0   0   0   0   1   1
R2   0   0   0   1   0   0   0   0   1   1   0   0
R3   0   0   0   0   0   1   1   1   0   0   0   0
    0      1      2      3      4      5      6      7      8      9      ...  \
R0      1      1      1      1      1      1      1      1      1      0  ...   
R1      1      1      1      1      1      1      1      1      1      0  ...   
R2      1      1      1      1      1      0      0      0      0      0  ...   
R3      1      0      0      0      0      0      0      0      0      0  ...   
T1      0      0      0      0      0      0      0      0      0      0  ...   
T2      0      0      0      0      0      0      0      0      0      0  ...   
T3      1      0      0      0      0      0      0      0      0      0  ...   
T4      0   

In [6]:
# Operazione per Time Transformer
time_intra_token = {
    (c1, c2): random.randint(1, 20) if c1 != tuple([[0]] * (num_tasks+num_regions)) else 0
    for c1 in unique_tuple_complete
    for c2 in unique_tuple_complete
}

times = [0] # Necessario come partenza di traccia (sotto partiamo dalla coppia in posizione 0 ed 1 con valore da assegnare alla posizione 1)
for i in range(1, len(df_traces_complete)):
    prev_tuple = tuple(df_traces_complete.iloc[i-1].astype(int))
    curr_tuple = tuple(df_traces_complete.iloc[i].astype(int))

    times.append(time_intra_token[(prev_tuple, curr_tuple)])

In [7]:
# Creo il tensor con le colonne codificate in interi
data_complete = torch.tensor(encode_complete(df_traces_complete.values), dtype=torch.long)
data_regions = torch.tensor(encode_regions(df_regions.values), dtype=torch.long)
data_tasks = torch.tensor(encode_tasks(df_tasks.values), dtype=torch.long)
data_times = torch.tensor(times, dtype=torch.long)

In [8]:
n = int(0.8 * len(df_traces_complete)) # 80% train, 20% validation

# Valori per il modello completo - REGIONE + TASK
block_size_mc = 256
n_embd_mc = 256
dropout_mc = 0.30
n_head_mc = 8
n_layer_mc = 3
batch_size_mc = 16
eval_iters_mc = 200
eval_interval_mc = 150
max_iters_mc = 1000
learning_rate_mc = 3e-4

# Creazione modello BPMNTransformer - Token --> REGIONE + TASK
model_complete = BPMNTransformer(vocab_size_complete, block_size_mc, n_embd_mc, dropout_mc, n_head_mc, n_layer_mc)
m_c = model_complete.to(device)

optimizer_m = torch.optim.AdamW(model_complete.parameters(), lr=learning_rate_mc) # Ottimizzatore pyTorch

train_data = data_complete[:n]
val_data = data_complete[n:]

# print(sum(p.numel() for p in m.parameters()) / 1e6, 'M parameters') # Calcolo parametri modelli

In [9]:
# Funzione presa dal video di karpathy (alleno il modello mc (modello completo - REGIONI + TASK) )
for iter in range(max_iters_mc):
    if iter % eval_interval_mc == 0 or iter == max_iters_mc - 1: # Ogni tot stampo la loss corrente
        losses = estimate_loss(model_complete, eval_iters_mc, train_data, val_data, batch_size_mc, block_size_mc, device)
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch_model('train', train_data, val_data, batch_size_mc, block_size_mc, device) # Pesco tracce

    logits, loss = model_complete(xb, yb) # Esegue il forward e predice
    optimizer_m.zero_grad(set_to_none=True) # Reset gradienti (puliamo i calcoli del giro precedente)

    loss.backward() #Errore per neurone
    optimizer_m.step() # Aggiorna automaticamente i pesi per sbagliare di meno al giro dopo

step 0: train loss 2.4716, val loss 2.4736
step 150: train loss 0.6212, val loss 0.6221
step 300: train loss 0.3882, val loss 0.3852
step 450: train loss 0.3512, val loss 0.3480
step 600: train loss 0.3472, val loss 0.3453
step 750: train loss 0.3471, val loss 0.3452
step 900: train loss 0.3461, val loss 0.3449
step 999: train loss 0.3457, val loss 0.3442


In [10]:
# Valori per il modello time
block_size_mt = 64
n_embd_mt = 128
dropout_mt = 0.30
n_head_mt = 8
n_layer_mt = 3
batch_size_mt = 16
eval_iters_mt = 200
eval_interval_mt = 150
max_iters_mt = 2000
learning_rate_mt = 3e-4

# Creazione modello TimeTransformer
time_model = TimeTransformer(vocab_size_complete, block_size_mt, n_embd_mt, dropout_mt, n_head_mt, n_layer_mt)
m_t = time_model.to(device)

optimizer_t = torch.optim.AdamW(time_model.parameters(), lr=learning_rate_mt) # Ottimizzatore pyTorch

train_times = data_times[:n]
val_times = data_times[n:]

In [11]:
# Funzione presa dal video di karpathy (alleno il modello del Tempo - adattata)
for iter in range(max_iters_mt):
    if iter % eval_interval_mt == 0 or iter == max_iters_mt - 1: # Ogni tot stampo la loss corrente
        losses = estimate_loss_times(time_model, eval_iters_mt, train_data, train_times, val_data, val_times, batch_size_mt, block_size_mt, device)
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xd, xt, y = get_batch_model_time('train', train_data, train_times, val_data, val_times, batch_size_mt, block_size_mt, device) # Pesco tracce

    logits, loss = time_model(xd, xt, y) # Esegue il forward e predice
    optimizer_t.zero_grad(set_to_none=True) # Reset gradienti (puliamo i calcoli del giro precedente)

    loss.backward() #Errore per neurone
    optimizer_t.step() # Aggiorna automaticamente i pesi per sbagliare di meno al giro dopo

step 0: train loss 126.2816, val loss 125.8494
step 150: train loss 46.4727, val loss 46.2494
step 300: train loss 18.0336, val loss 17.5507
step 450: train loss 10.5156, val loss 10.0955
step 600: train loss 5.6359, val loss 5.5113
step 750: train loss 3.1149, val loss 2.8708
step 900: train loss 1.7760, val loss 1.6240
step 1050: train loss 1.0177, val loss 0.9662
step 1200: train loss 0.9747, val loss 0.8952
step 1350: train loss 0.8362, val loss 0.7737
step 1500: train loss 0.8788, val loss 0.8062
step 1650: train loss 1.1947, val loss 1.1268
step 1800: train loss 0.8949, val loss 0.8119
step 1950: train loss 0.7387, val loss 0.6947
step 1999: train loss 0.5217, val loss 0.5185


In [12]:
# Valori per il modello completo - SOLO TASK
block_size_mtask = 256
n_embd_mtask = 256
dropout_mtask = 0.30
n_head_mtask = 8
n_layer_mtask = 3
batch_size_mtask = 16
eval_iters_mtask = 200
eval_interval_mtask = 150
max_iters_mtask = 1000
learning_rate_mtask = 3e-4

# Creazione modello BPMNTransformer - Token --> SOLO TASK
model_task = BPMNTransformer(vocab_size_tasks, block_size_mtask, n_embd_mtask, dropout_mtask, n_head_mtask, n_layer_mtask)
m_task = model_task.to(device)

optimizer_mtask = torch.optim.AdamW(model_task.parameters(), lr=learning_rate_mtask) # Ottimizzatore pyTorch

train_data_task = data_tasks[:n]
val_data_task = data_tasks[n:]

In [13]:
# Funzione presa dal video di karpathy (alleno il modello mc (modello completo - REGIONI + TASK) )
for iter in range(max_iters_mtask):
    if iter % eval_interval_mtask == 0 or iter == max_iters_mtask - 1: # Ogni tot stampo la loss corrente
        losses = estimate_loss(model_task, eval_iters_mtask, train_data_task, val_data_task, batch_size_mtask, block_size_mtask, device)
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch_model('train', train_data_task, val_data_task, batch_size_mtask, block_size_mtask, device) # Pesco tracce

    logits, loss = model_task(xb, yb) # Esegue il forward e predice
    optimizer_mtask.zero_grad(set_to_none=True) # Reset gradienti (puliamo i calcoli del giro precedente)

    loss.backward() #Errore per neurone
    optimizer_mtask.step() # Aggiorna automaticamente i pesi per sbagliare di meno al giro dopo

step 0: train loss 2.2621, val loss 2.2620
step 150: train loss 0.6614, val loss 0.6589
step 300: train loss 0.3558, val loss 0.3531
step 450: train loss 0.3524, val loss 0.3494
step 600: train loss 0.3489, val loss 0.3467
step 750: train loss 0.3483, val loss 0.3470
step 900: train loss 0.3500, val loss 0.3476
step 999: train loss 0.3483, val loss 0.3466


In [14]:
# Valori per il modello completo - SOLO TASK
block_size_mregion = 256
n_embd_mregion = 256
dropout_mregion = 0.30
n_head_mregion = 8
n_layer_mregion = 3
batch_size_mregion = 16
eval_iters_mregion = 200
eval_interval_mregion = 150
max_iters_mregion = 1000
learning_rate_mregion = 3e-4

# Creazione modello BPMNTransformer - Token --> SOLO TASK
model_region = RegionTransformer(vocab_size_regions, vocab_size_tasks, block_size_mregion, n_embd_mregion, dropout_mregion, n_head_mregion, n_layer_mregion)
m_region = model_region.to(device)

optimizer_mregion = torch.optim.AdamW(model_region.parameters(), lr=learning_rate_mregion) # Ottimizzatore pyTorch

train_data_region = data_regions[:n]
val_data_region = data_regions[n:]

In [15]:
# Funzione presa dal video di karpathy (alleno il modello mc (modello completo - REGIONI + TASK) )
for iter in range(max_iters_mregion):
    if iter % eval_interval_mregion == 0 or iter == max_iters_mregion - 1: # Ogni tot stampo la loss corrente
        losses = estimate_loss_region(model_region, eval_iters_mregion, train_data_task, train_data_region, val_data_task, val_data_region, batch_size_mregion, block_size_mregion, device)
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    x_task, x_region, y = get_batch_model_region('train', train_data_task, val_data_task, train_data_region, val_data_region, batch_size_mregion, block_size_mregion, device) # Pesco tracce

    logits, loss = model_region(x_region, x_task, y) # Esegue il forward e predice
    optimizer_mregion.zero_grad(set_to_none=True) # Reset gradienti (puliamo i calcoli del giro precedente)

    loss.backward() #Errore per neurone
    optimizer_mregion.step() # Aggiorna automaticamente i pesi per sbagliare di meno al giro dopo

step 0: train loss 1.7061, val loss 1.7081
step 150: train loss 0.1264, val loss 0.1257
step 300: train loss 0.0169, val loss 0.0157
step 450: train loss 0.0058, val loss 0.0058
step 600: train loss 0.0077, val loss 0.0067
step 750: train loss 0.0033, val loss 0.0034
step 900: train loss 0.0076, val loss 0.0079
step 999: train loss 0.0032, val loss 0.0038


In [16]:
# Valori per il modello time v2
block_size_mt_v2 = 64
n_embd_mt_v2 = 128
dropout_mt_v2 = 0.30
n_head_mt_v2 = 8
n_layer_mt_v2 = 3
batch_size_mt_v2 = 16
eval_iters_mt_v2 = 200
eval_interval_mt_v2 = 150
max_iters_mt_v2 = 2000
learning_rate_mt_v2 = 3e-4

# Creazione modello TimeTransformerV2
time_model_v2 = TimeTransformerV2(vocab_size_tasks, vocab_size_regions, block_size_mt_v2, n_embd_mt_v2, dropout_mt_v2, n_head_mt_v2, n_layer_mt_v2)
m_t_v2 = time_model_v2.to(device)

optimizer_t_v2 = torch.optim.AdamW(time_model_v2.parameters(), lr=learning_rate_mt_v2) # Ottimizzatore pyTorch

train_times = data_times[:n]
val_times = data_times[n:]

In [17]:
# Funzione presa dal video di karpathy (alleno il modello del Tempo - adattata per la versione V2 del TimeTransformer (task e regioni separate))
for iter in range(max_iters_mt_v2):
    if iter % eval_interval_mt_v2 == 0 or iter == max_iters_mt_v2 - 1: # Ogni tot stampo la loss corrente
        losses = estimate_loss_times_v2(time_model_v2, eval_iters_mt_v2, train_data_task, train_data_region, train_times, val_data_task, val_data_region, val_times, batch_size_mt_v2, block_size_mt_v2, device)
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xta, xr, xt, y = get_batch_model_time_v2('train', train_data_task, train_data_region, train_times, val_data_task, val_data_region, val_times, batch_size_mt_v2, block_size_mt_v2, device) # Pesco tracce

    logits, loss = time_model_v2(xta, xr, xt, y) # Esegue il forward e predice
    optimizer_t_v2.zero_grad(set_to_none=True) # Reset gradienti (puliamo i calcoli del giro precedente)

    loss.backward() #Errore per neurone
    optimizer_t_v2.step() # Aggiorna automaticamente i pesi per sbagliare di meno al giro dopo

step 0: train loss 126.3558, val loss 125.9204
step 150: train loss 46.3270, val loss 45.9872
step 300: train loss 19.0815, val loss 18.7743
step 450: train loss 11.8943, val loss 11.6420
step 600: train loss 5.8323, val loss 5.6274
step 750: train loss 5.1870, val loss 4.9741
step 900: train loss 3.5283, val loss 3.3272
step 1050: train loss 2.0533, val loss 1.8605
step 1200: train loss 1.2674, val loss 1.1654
step 1350: train loss 0.9959, val loss 0.9063
step 1500: train loss 0.8856, val loss 0.8084
step 1650: train loss 0.8237, val loss 0.7642
step 1800: train loss 0.8314, val loss 0.7597
step 1950: train loss 0.6176, val loss 0.5522
step 1999: train loss 0.7087, val loss 0.6721


In [18]:
# Generazione - REGIONE+TASK --> Time

context = torch.tensor(encode_complete([[0]*(num_regions+num_tasks)]), dtype=torch.long, device=device).unsqueeze(0)
context_time = torch.tensor([0], dtype=torch.float32, device=device).unsqueeze(0) # Lo zero sembra corretto al momento (da verificare, sembrerebbe andare bene anche 1)

# Faccio indovinare il primo time (per (0,0,0,...,0) --> cosi prende meglio il contesto, altrimenti inizia a cannare i primi valori)
first_time_guess = m_t.predict_next_time(idx_tasks=context, idx_times=context_time, block_size=block_size_mt)
first_time_guess = torch.round(first_time_guess)
context_time = torch.cat((context_time, first_time_guess), dim=1)

max_new_tokens = 30
generated_indices_completegen = []
generated_times_completegen = []
for step in range(max_new_tokens):
    # CON MASCHERA 0 FINALE
    '''next_id = m.predict_next_task(idx=context, block_size=block_size)
    context = torch.cat((context, next_id), dim=1)
    context_time_fake = torch.cat((context_time, fake_time), dim=1)
    next_time = t.predict_next_time(idx_tasks=context, idx_times=context_time_fake, block_size=block_size)
    next_time = torch.round(next_time)
    context_time = torch.cat((context_time, next_time), dim=1)
    generated_indices.append(next_id.item())
    generated_times.append(next_time.item())'''

    # CON SLIDING WINDOW
    next_id = m_c.predict_next_task(idx=context, block_size=block_size_mc)
    context = torch.cat((context, next_id), dim=1)
    next_time = m_t.predict_next_time(idx_tasks=context, idx_times=context_time, block_size=block_size_mt)
    next_time = torch.round(next_time)
    context_time = torch.cat((context_time, next_time), dim=1)
    generated_indices_completegen.append(next_id.item())
    generated_times_completegen.append(next_time.item())

decoded_output_complete_firstgen = decode_complete(generated_indices_completegen)

for i in range(len(decoded_output_complete_firstgen)):
    step = decoded_output_complete_firstgen[i]
    bit_list = [int(b) for b in step]

    time = generated_times_completegen[i]

    if i>0:
        real_time = time_intra_token[tuple(decoded_output_complete_firstgen[i-1]),tuple(decoded_output_complete_firstgen[i])]
    else:
        real_time = time_intra_token[tuple([0]*(num_regions+num_tasks)),tuple(decoded_output_complete_firstgen[i])]
    print(f"{bit_list} - {time} - {real_time}")

[1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0] - 0.0 - 0
[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 4.0 - 4
[1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0] - 10.0 - 10
[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 4.0 - 5
[1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0] - 10.0 - 10
[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 13.0 - 11
[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0] - 14.0 - 14
[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 17.0 - 17
[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1] - 7.0 - 7
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 6.0 - 6
[1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0] - 0.0 - 0
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 18.0 - 18
[1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0] - 0.0 - 0
[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 17.0 - 16
[1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0] - 17.0 - 17
[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 4.0 - 4
[1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0] - 11.0 - 10
[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 10.0 - 11
[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0] - 14.0 - 14
[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 17.0 - 17
[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]

In [19]:
# Funzione per unire Task e Region in un ID compatibile col TimeTransformer --> ADESSO IN TEORIA NON SERVE PIU'
def get_complete_token(task_tensor, region_tensor):
    t_int = task_tensor.item()
    r_int = region_tensor.item()

    t_decoded = decode_tasks([t_int])[0]
    r_decoded = decode_regions([r_int])[0]

    decoding = [int(b) for b in r_decoded] + [int(b) for b in t_decoded]

    try:
        c_encoded = encode_complete([decoding])[0]
        return True, torch.tensor([[c_encoded]], dtype=torch.long, device=device)
    except:
        return False, decoding

In [24]:
# Generazione - TASK --> REGIONE --> Time
context_task = torch.tensor(encode_tasks([[0]*num_tasks]), dtype=torch.long, device=device).unsqueeze(0)
context_region = torch.tensor(encode_regions([[0]*num_regions]), dtype=torch.long, device=device).unsqueeze(0)
context_time = torch.tensor([0], dtype=torch.float32, device=device).unsqueeze(0) # Lo zero sembra corretto al momento (da verificare, sembrerebbe andare bene anche 1)

max_new_tokens = 30
generated_indices_task = []
generated_indices_region = []
generated_times_partgen = []
for step in range(max_new_tokens):
    next_task = m_task.predict_next_task(idx=context_task, block_size=block_size_mtask)
    context_task = torch.cat((context_task, next_task), dim=1)

    context_task_aligned = context_task[:, 1:]

    next_region = m_region.predict_next_region(idx_region=context_region, idx_task=context_task_aligned, block_size=block_size_mregion)
    context_region = torch.cat((context_region, next_region), dim=1)

    context_region_aligned = context_region[:, 1:]

    next_time = m_t_v2.predict_next_time(idx_tasks=context_task_aligned, idx_regions=context_region_aligned, idx_times=context_time, block_size=block_size_mt_v2)
    next_time = torch.round(next_time)
    context_time = torch.cat((context_time, next_time), dim=1)

    generated_indices_task.append(next_task.item())
    generated_indices_region.append(next_region.item())
    generated_times_partgen.append(next_time.item())

for i in range(max_new_tokens):
    t_int = generated_indices_task[i]
    r_int = generated_indices_region[i]

    t_bits = [int(b) for b in decode_tasks([t_int])[0]]
    r_bits = [int(b) for b in decode_regions([r_int])[0]]

    bit_list = r_bits + t_bits

    time = generated_times_partgen[i]

    if i > 0:
        past_t_bits = [int(b) for b in decode_tasks([generated_indices_task[i-1]])[0]]
        past_r_bits = [int(b) for b in decode_regions([generated_indices_region[i-1]])[0]]
        past_bit_list = past_r_bits + past_t_bits

        key = (tuple(past_bit_list), tuple(bit_list))
    else:
        key = (tuple([0]*(num_regions+num_tasks)), tuple(bit_list))

    if key in time_intra_token:
        real_time = time_intra_token[key]
    else:
        real_time = None

    print(f"{bit_list} - {time} - {real_time}")


[1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0] - 11.0 - 0
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 6.0 - 18
[1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0] - 0.0 - 0
[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 5.0 - 5
[1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0] - 17.0 - 17
[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 4.0 - 4
[1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0] - 10.0 - 10
[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 12.0 - 11
[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0] - 14.0 - 14
[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 17.0 - 17
[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1] - 7.0 - 7
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 6.0 - 6
[1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0] - 0.0 - 0
[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 5.0 - 5
[1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0] - 10.0 - 10
[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 19.0 - 20
[1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0] - 17.0 - 17
[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 12.0 - 11
[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0] - 14.0 - 14
[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 17.0 - 17
[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]

In [119]:
print("\n--- TRACCE GENERATE ---")
traces_generated = []
current_trace_generated = []
for i, step in enumerate(decoded_output):
    bit_list = [int(b) for b in step]
    current_trace_generated.append(bit_list)
    if bit_list == [0]*(num_regions+num_tasks):
        traces_generated.append(current_trace_generated)
        current_trace_generated = []
    #print(f"Step {i:02d}: {bit_list}")

traces_generated.remove(traces_generated[0]) #Rimuovo la prima che è sempre [], generata ed inserita dall'algoritmo sopra
for i,trace in enumerate(traces_generated):
    print(f"{i}: {trace}")


[1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0] - 0.0 - 0
[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0] - 18.0 - 12
[1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0] - 4.0 - 12
[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 13.0 - 16
[1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0] - 2.0 - 14
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 15.0 - 12
[1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1] - 19.0 - 0
[1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0] - 5.0 - 7
[1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0] - 1.0 - 13
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 14.0 - 10
[1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0] - 0.0 - 0
[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0] - 12.0 - 12
[1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0] - 14.0 - 12
[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0] - 19.0 - 12
[1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0] - 2.0 - 12
[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 15.0 - 16
[1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0] - 19.0 - 14
[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] - 5.0 - 3
[1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0] - 1.0 - 1
[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0] - 14.0 - 12
[1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0

In [12]:
from pm4py.algo.conformance.alignments.petri_net import algorithm as alignments
from pm4py.objects.log.obj import Trace, Event, EventLog

# Facciamo il decoding delle tracce generate
#print(traces_generated)
traces_decoded = getDecoding(traces_generated, net.regions, net.tasks)
for i,trace in enumerate(traces_decoded):
    print(f"{i}: {trace}")

# Creiamo il l'EventLog di ogni traccia per poi poterla allineare
eventlog_traces = EventLog()
for trace in traces_decoded:
    t = Trace()
    for activity in trace:
        t.append(Event({'concept:name': activity}))
    eventlog_traces.append(t)


check = ["P", "X", "L"]
start = tuple(["start_" + c for c in check])
end = tuple(["end_" + c for c in check if c!="L"]) # si escludono gli end loop
loop = tuple(["back_L"]) + tuple(["end_L"])

silent_transition_prefixes = start + end + loop

model_cost_function = dict()
sync_cost_function = dict()

# METTIAMO 10000 come di default???
for t in net.net.transitions: # Prendo tutte le transizioni
    if t.label is not None and t.label.startswith(silent_transition_prefixes): # Se è una transizione silente,
        model_cost_function[t] = 0
        sync_cost_function[t] = 10000

    elif t.label is None:
        model_cost_function[t] = 0
        sync_cost_function[t] = 10000

    else: # Se è un task vero e proprio
        model_cost_function[t] = 10000
        sync_cost_function[t] = 0

parameters = {
    alignments.Parameters.PARAM_MODEL_COST_FUNCTION: model_cost_function,
    alignments.Parameters.PARAM_SYNC_COST_FUNCTION: sync_cost_function
}

# Eseguiamo l'allineamento
aligned_traces = alignments.apply(eventlog_traces, net.net, net.initial_marking, net.final_marking, parameters=parameters)
for i,trace in enumerate(aligned_traces):
    print(f"{i}: {trace}")

0: ['start_T1', 'end_T1', 'start_T2', 'end_T2', 'start_T5', 'end_T5', 'start_T3', 'end_T3', 'start_T4', 'end_T4', 'start_T6', 'end_T6', 'start_T6', 'end_T6', 'start_T2', 'end_T2', 'start_T5', 'end_T5', 'start_T1', 'end_T1', 'start_T6', 'end_T6', 'start_T5', 'end_T5', 'start_T4', 'end_T4', 'start_T6', 'end_T6', 'start_T3', 'end_T3', 'start_T6', 'end_T6', 'start_T6', 'end_T6', 'start_T5', 'end_T5', 'start_T4', 'end_T4', 'start_T3', 'end_T3', 'start_T6', 'end_T6', 'start_T3', 'end_T3', 'start_T3', 'end_T3', 'start_T3', 'end_T3', 'start_T3', 'end_T3', 'start_T5', 'end_T5', 'start_T6', 'end_T6', 'start_T1', 'end_T1', 'start_T2', 'end_T2', 'start_T1', 'end_T1', 'start_T1', 'end_T1', 'start_T6', 'end_T6', 'start_T6', 'end_T6', 'start_T2', 'end_T2', 'start_T5', 'end_T5', 'start_T1', 'end_T1', 'start_T4', 'end_T4', 'start_T3', 'end_T3', 'start_T4', 'end_T4', 'start_T6', 'end_T6', 'start_T6', 'end_T6', 'start_T4', 'end_T4', 'start_T6', 'end_T6', 'start_T6', 'end_T6', 'start_T1', 'end_T1', 'start

aligning log, completed variants ::   0%|          | 0/12 [00:00<?, ?it/s]

0: {'alignment': [('>>', 'start_L0'), ('>>', 'start_P1'), ('>>', 'start_L8'), ('>>', 'start_L3'), ('>>', 'start_P4'), ('start_T1', 'start_T1'), ('end_T1', 'end_T1'), ('start_T2', 'start_T2'), ('end_T2', 'end_T2'), ('start_T5', 'start_T5'), ('end_T5', 'end_T5'), ('>>', 'end_P4'), ('>>', 'end_L3'), ('>>', 'start_L5'), ('>>', 'start_P6'), ('>>', 'start_L7'), ('start_T3', 'start_T3'), ('end_T3', 'end_T3'), ('start_T4', 'start_T4'), ('end_T4', 'end_T4'), ('>>', 'end_L7'), ('>>', 'end_P6'), ('>>', 'start_L10'), ('>>', 'end_L5'), ('start_T6', 'start_T6'), ('end_T6', 'end_T6'), ('>>', 'back_L10'), ('start_T6', 'start_T6'), ('end_T6', 'end_T6'), ('>>', 'end_L10'), ('>>', 'end_L8'), ('>>', 'end_P1'), ('>>', 'back_L0'), ('>>', 'start_P1'), ('>>', 'start_L3'), ('>>', 'start_P4'), ('>>', 'start_L8'), ('start_T2', 'start_T2'), ('end_T2', 'end_T2'), ('start_T5', 'start_T5'), ('end_T5', 'end_T5'), ('start_T1', 'start_T1'), ('>>', 'start_L10'), ('end_T1', 'end_T1'), ('>>', 'end_P4'), ('>>', 'end_L3'), 

In [13]:
check = ["P", "X", "L"]
start = tuple(["start_" + c for c in check])
end = tuple(["end_" + c for c in check if c!="L"]) # si escludono gli end loop
back_loop = tuple(["back_L"])

# Puliamo l'allineamento e otteniamo la traccia allineata più vicina a quella generata
aligned_traces_cleaned = []
for a_trace in aligned_traces:
    trace_cleaned = []
    trace = a_trace['alignment']

    print(trace)

    for _, net_step in trace: #trans --> trans step , net --> real petri net step
        if net_step != None:
            if not (net_step.startswith(start) or net_step.startswith(end) or net_step.startswith(back_loop)) and not net_step=='>>':
                trace_cleaned.append(net_step)

    aligned_traces_cleaned.append(trace_cleaned)


for i,trace in enumerate(aligned_traces_cleaned):
    print(f"{i}: {trace}")

# Codifichiamo le tracce allineati (per poi poterle confrontare con quelle generate dal transformer)
aligned_traceEncoded_regions, aligned_traceEncoded_tasks = getEncoding(aligned_traces_cleaned, net.regions, net.tasks, net.open_clauses, net.end_clauses)
df_aligned_traces = pd.concat([aligned_traceEncoded_regions, aligned_traceEncoded_tasks], axis=0)
print(df_aligned_traces)


[('>>', 'start_L0'), ('>>', 'start_P1'), ('>>', 'start_L8'), ('>>', 'start_L3'), ('>>', 'start_P4'), ('start_T1', 'start_T1'), ('end_T1', 'end_T1'), ('start_T2', 'start_T2'), ('end_T2', 'end_T2'), ('start_T5', 'start_T5'), ('end_T5', 'end_T5'), ('>>', 'end_P4'), ('>>', 'end_L3'), ('>>', 'start_L5'), ('>>', 'start_P6'), ('>>', 'start_L7'), ('start_T3', 'start_T3'), ('end_T3', 'end_T3'), ('start_T4', 'start_T4'), ('end_T4', 'end_T4'), ('>>', 'end_L7'), ('>>', 'end_P6'), ('>>', 'start_L10'), ('>>', 'end_L5'), ('start_T6', 'start_T6'), ('end_T6', 'end_T6'), ('>>', 'back_L10'), ('start_T6', 'start_T6'), ('end_T6', 'end_T6'), ('>>', 'end_L10'), ('>>', 'end_L8'), ('>>', 'end_P1'), ('>>', 'back_L0'), ('>>', 'start_P1'), ('>>', 'start_L3'), ('>>', 'start_P4'), ('>>', 'start_L8'), ('start_T2', 'start_T2'), ('end_T2', 'end_T2'), ('start_T5', 'start_T5'), ('end_T5', 'end_T5'), ('start_T1', 'start_T1'), ('>>', 'start_L10'), ('end_T1', 'end_T1'), ('>>', 'end_P4'), ('>>', 'end_L3'), ('start_T6', 'sta

In [14]:
# Creo una lista delle tracce codificate (ogni traccia è una lista dove ogni elemento è una colonna del df, ossia uno step) --> più facili da confrontare quando calcoliamo la distanza
aligned_traces_encoded = []
aligned_trace_encoded = []
for element in df_aligned_traces.T.values:
    element = element.tolist()
    aligned_trace_encoded.append(element)
    if element == [0] * (num_regions+num_tasks):
        aligned_traces_encoded.append(aligned_trace_encoded)
        aligned_trace_encoded = []

# Andiamo a calcolare il costo con la edit distance (weighted_levenshtein)
for i in range(len(aligned_traces_encoded)):
    generated_t = traces_generated[i]
    aligned_t = aligned_traces_encoded[i]

    gen_tuples = [tuple(step) for step in generated_t]
    aligned_tuples = [tuple(step) for step in aligned_t]

    total_possible_column = set(gen_tuples + aligned_tuples) # Prendo tutte le possibili colonne per andare a creare il dizionario delle sostituzioni

    # Se volessi dizionario delle distanze bisognerebbe usare questo pezzo di codice (adesso usiamo distanza di hamming di base nel codice)
    '''cost_sub = {}
    for e1 in total_possible_column:
        for e2 in total_possible_column:
            if e1 != e2:
                counter = 0
                for i in range(len(e1)):
                    if e1[i] != e2[i]:
                        counter+=1
                cost_sub[(e1,e2)] = counter'''

    cost = edit_distance_weighted_levenshtein(generated_t, aligned_t, num_regions+num_tasks, num_regions+num_tasks, hamming_distance) # Utilizziamo la distanza di hamming al momento
    print(cost)

#costo = edit_distance_weighted_levenshtein(traccia_ai, traccia_pulita, cost_ins, cost_del, cost_sub)
#print(f"Costo di correzione totale: {costo}")

185.0
37.0
156.0
76.0
93.0
168.0
38.0
174.0
80.0
18.0
66.0
43.0


In [14]:
print()